In [1]:
from sqlalchemy import create_engine

import folium
import math
import numpy as np
import pandas as pd
import pymysql
import rdflib as rdf

In [2]:
engine = create_engine(open("conn.txt", "r").read())

In [3]:
q =  "SELECT * FROM vocabulary"
vocabs_os_df = pd.read_sql_query(q, engine)
vocabs_os_df.set_index('id', inplace=True)

q = "SELECT * FROM property"
props_os_df = pd.read_sql_query(q, engine).merge(vocabs_os_df, left_on = 'vocabulary_id', right_on='id')

props_os_df['predicate'] = props_os_df[['namespace_uri', 'local_name']].apply(lambda x: ''.join(x), axis=1)

print(f'Length of df:{len(props_os_df)}')

Length of df:202


In [4]:
q = "SELECT * FROM value"

items_tmp_df = pd.read_sql_query(q, engine).merge(props_os_df, left_on = 'property_id', right_on = 'id',
                                                suffixes=('_value', '_vocab'))

print(f'Length of df:{len(items_tmp_df)}')

Length of df:11914


In [5]:
items_df_strings = items_tmp_df[['resource_id','predicate','value_resource_id','value', 'uri']].merge(items_tmp_df.query('predicate == "http://purl.org/dc/terms/identifier"')[['resource_id','value']],
   left_on = 'resource_id', right_on = 'resource_id', how = 'left')

items_df_strings.rename(columns={'value_y':'subject', 'value_x':'object_str'},inplace=True)
items_df_strings.drop(columns='resource_id', inplace=True)

print(f'Length of df:{len(items_df_strings)}')

Length of df:11914


In [6]:
# just a further bit of checking
items_df_strings.predicate.unique()

array(['http://purl.org/dc/terms/title',
       'http://purl.org/dc/terms/description',
       'http://purl.org/dc/terms/identifier',
       'http://www.w3.org/2000/01/rdf-schema#subClassOf',
       'http://www.w3.org/1999/02/22-rdf-syntax-ns#type',
       'http://p-lod.umasscreate.net/vocabulary#spatially-within',
       'http://p-lod.umasscreate.net/vocabulary#p-in-p-url',
       'http://p-lod.umasscreate.net/vocabulary#arcgis-id',
       'http://p-lod.umasscreate.net/vocabulary#surface-area',
       'http://p-lod.umasscreate.net/vocabulary#pleiades-url',
       'http://p-lod.umasscreate.net/vocabulary#getty-lod-url',
       'http://p-lod.umasscreate.net/vocabulary#wikidata-url',
       'http://p-lod.umasscreate.net/vocabulary#found-within',
       'http://p-lod.umasscreate.net/vocabulary#created-on-surface-of',
       'http://www.w3.org/2000/01/rdf-schema#seeAlso',
       'http://p-lod.umasscreate.net/vocabulary#has-pompeian-wall-painting-style',
       'http://p-lod.umasscreate.net

In [7]:
items_df_things = items_df_strings.merge(items_tmp_df.query('predicate == "http://purl.org/dc/terms/identifier"')[['resource_id','value']],
   left_on = 'value_resource_id', right_on = 'resource_id', how = 'left')

items_df_things.rename(columns={'value':'object_thing'},inplace=True)
items_df_things.drop(columns='value_resource_id', inplace=True)

items_df = items_df_things[['subject','predicate', 'object_thing', 'object_str', 'uri']]

print(f'Length of df:{len(items_df)}')

items_df[items_df.subject.isnull()]

Length of df:11914


,subject,predicate,object_thing,object_str,uri
0,NaN,http://purl.org/dc/terms/title,NaN,P-LOD Classes,None
1,NaN,http://purl.org/dc/terms/title,NaN,P-LOD Items,None
1428,NaN,http://purl.org/dc/terms/description,NaN,Classes for use by Pompeii Linked Open Data.,None
1429,NaN,http://purl.org/dc/terms/description,NaN,Omeka S items that identify and define entitie...,None


In [8]:
# only 4 rows should have been output above
items_df = items_df[items_df.subject.notnull()]
print(f'Length of df:{len(items_df)}')

Length of df:11910


In [9]:
G = rdf.Graph()

In [10]:
for i,r in items_df.iterrows():
    if str((r['object_thing'])) != 'nan':
        G.add((rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r.subject}'),
               rdf.URIRef(r.predicate),
               rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r.object_thing}')))
    elif r['uri'] != None:
        if r['object_str'] != None:
            G.add((rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r.subject}'),
               rdf.URIRef(r.predicate),
               rdf.URIRef(r.uri)))
        elif r['object_str'] == None:
            G.add((rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r.subject}'),
               rdf.URIRef(r.predicate),
               rdf.URIRef(r.uri)))
    elif r['object_str'] != None:
        G.add((rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r.subject}'),
               rdf.URIRef(r.predicate),
               rdf.Literal(r.object_str)))

In [11]:
# in progress
q = "SELECT * FROM mapping_marker"

df = pd.read_sql_query(q, engine)

mm_df = df.merge(items_tmp_df.query('predicate == "http://purl.org/dc/terms/identifier"')[['resource_id','value']],
            left_on = 'item_id',
            right_on = 'resource_id', how = 'left')\
            [['value','geojson']].rename(columns = {'value':'subject'})

In [12]:
for i,r in mm_df.iterrows():
    G.add((rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r.subject}'),
               rdf.URIRef(u'http://p-lod.umasscreate.net/vocabulary#geojson'),
               rdf.Literal(r.geojson)))

In [13]:
# in progress
q = "SELECT * FROM resource WHERE resource_class_id = 151"

df = pd.read_sql_query(q, engine)

class_df = df.merge(items_tmp_df.query('predicate == "http://purl.org/dc/terms/identifier"')[['resource_id','value']],
            left_on = 'id',
            right_on = 'resource_id', how = 'left')\
            ['value'].rename(columns = {'value':'subject'})

In [14]:
for r in class_df:
    G.add((rdf.URIRef(f'http://p-lod.umasscreate.net/vocabulary#{r}'),
               rdf.URIRef(u'http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
               rdf.URIRef(u'http://www.w3.org/2002/07/owl#Class')))

In [15]:
ns_dict = dict(zip(vocabs_os_df.prefix,vocabs_os_df.namespace_uri))

for k in ns_dict.keys():
    G.bind(k,ns_dict[k])

In [16]:
ttl = G.serialize(destination = None, format="turtle").decode()
with open("p-lod.ttl", "w") as file:
    file.write(ttl)

In [17]:
# confirm that .ttl file is loadable back into rdflib. seems like a good idea.
G = rdf.Graph()
G.parse("p-lod.ttl", format = "turtle")



<Graph identifier=N285149e716b04cd89720dee50e101147 (<class 'rdflib.graph.Graph'>)>

In [18]:
# and now query it

"SELECT ?s ?o WHERE {?s <http://p-lod.umasscreate.net/vocabulary#geojson> ?o}"

result = G.query("SELECT ?s ?o WHERE {?s <http://p-lod.umasscreate.net/vocabulary#geojson> ?o}")
len(result)

114